In [4]:
#PREPARE RAG FORM AZURE WELL ARCHITETED FRAMEWORK

import unicodedata
from pathlib import Path

from pypdf import PdfReader
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def prepare_index_pdf():

    embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        cache_folder="./models",
        device="cpu",
    )

    Settings.embed_model = embed_model
    Settings.chunk_size = 900
    Settings.chunk_overlap = 150

    def clean_text(text: str) -> str:
        text = text.replace("\x00", " ")
        text = unicodedata.normalize("NFKD", text)
        text = text.encode("ascii", errors="ignore").decode("ascii")
        return " ".join(text.split())

    pdf_path = Path("data/import_pdf/azure-well-architected-ai.pdf")
    storage_path = Path("data/storage/azure-well-architected-ai-local")

    reader = PdfReader(str(pdf_path))

    documents = []

    for page_number, page in enumerate(reader.pages, start=1):
        raw_text = page.extract_text() or ""
        text = clean_text(raw_text)

        if text:
            documents.append(
                Document(
                    text=text,
                    metadata={
                        "file_name": pdf_path.name,
                        "page": page_number,
                    },
                )
            )

    print(f"Page count: {len(documents)}")

    index = VectorStoreIndex.from_documents(
        documents,
        show_progress=True,
    )

    index.storage_context.persist(
        persist_dir=str(storage_path)
    )

    print(f"Index saved: {storage_path}")

prepare_index_pdf()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22685.47it/s]


Page count: 138


Generating embeddings: 100%|██████████| 138/138 [00:03<00:00, 36.27it/s]


Index saved: data/storage/azure-well-architected-ai-local
